<a href="https://colab.research.google.com/github/dipti-2211/Cloud9/blob/main/Cloud9ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install rasterio geopandas rasterstats shapely fiona osmnx xarray netCDF4 -q


In [35]:
import osmnx as ox
import geopandas as gpd

DISTRICT_NAME = "Dima Hasao, Assam, India"
boundary = ox.geocode_to_gdf(DISTRICT_NAME)
boundary.to_file("boundary.geojson", driver="GeoJSON")

print("Boundary CRS:", boundary.crs)
print("Boundary bounds:", boundary.total_bounds)

Boundary CRS: epsg:4326
Boundary bounds: [92.5205147 24.9710763 93.4734857 25.8289431]


In [36]:
import rasterio
from rasterio.mask import mask

with rasterio.open("DEM.tif") as src:
    print("DEM CRS:", src.crs)
    print("DEM bounds:", src.bounds)
    clipped, transform = mask(src, boundary.geometry, crop=True)
    meta = src.meta.copy()
    meta.update({"height": clipped.shape[1], "width": clipped.shape[2], "transform": transform})

with rasterio.open("dem_clipped.tif", "w", **meta) as dst:
    dst.write(clipped)

print("Clipped OK — if this ran without a WindowError, DEM and boundary now overlap.")

DEM CRS: EPSG:4326
DEM bounds: BoundingBox(left=92.94986109999999, bottom=25.150138899999995, right=93.0998611, top=25.300138899999993)
Clipped OK — if this ran without a WindowError, DEM and boundary now overlap.


In [37]:
from rasterio.warp import calculate_default_transform, reproject, Resampling

dst_crs = "EPSG:32646"

with rasterio.open("dem_clipped.tif") as src:
    transform, width, height = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
    meta = src.meta.copy()
    meta.update({"crs": dst_crs, "transform": transform, "width": width, "height": height})
    with rasterio.open("dem_reprojected.tif", "w", **meta) as dst:
        reproject(
            source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear
        )

print("Reprojected to UTM 46N.")

Reprojected to UTM 46N.


In [38]:
import numpy as np

with rasterio.open("dem_reprojected.tif") as src:
    dem_arr = src.read(1).astype(float)
    dem_transform = src.transform
    dem_meta = src.meta.copy()
    nodata = src.nodata

if nodata is not None:
    dem_arr[dem_arr == nodata] = np.nan

px_width = dem_transform.a       # meters per pixel, x direction
px_height = -dem_transform.e     # meters per pixel, y direction (positive)

# Gradient: axis 0 = rows (north-south), axis 1 = cols (east-west)
dzdy, dzdx = np.gradient(dem_arr, px_height, px_width)

slope_rad = np.arctan(np.sqrt(dzdx**2 + dzdy**2))
slope_deg = np.degrees(slope_rad)

aspect_rad = np.arctan2(dzdy, -dzdx)
aspect_deg = (90 - np.degrees(aspect_rad)) % 360

def save_raster(path, array, meta):
    meta = meta.copy()
    meta.update(dtype="float32", count=1)
    with rasterio.open(path, "w", **meta) as dst:
        dst.write(array.astype("float32"), 1)

save_raster("slope.tif", slope_deg, dem_meta)
save_raster("aspect.tif", aspect_deg, dem_meta)

print("Slope/aspect computed. Slope range:", np.nanmin(slope_deg), "-", np.nanmax(slope_deg), "degrees")

Slope/aspect computed. Slope range: 0.028982813985941675 - 57.61313031547604 degrees


In [42]:
import os

if os.path.exists("gsi_full_inventory_cache.csv"):
    df = pd.read_csv("gsi_full_inventory_cache.csv")
    print("Loaded from cache:", len(df), "rows")
else:
    # ... your existing pdfplumber extraction code that builds df ...
    df.to_csv("gsi_full_inventory_cache.csv", index=False)

In [ ]:
# ============================================================
# CELL 7 (replacement): Extract real landslide points from the PDF, filtered to Dima Hasao
# ============================================================
import pdfplumber
import pandas as pd

col_names = ['Sl_No', 'Slide_No', 'State', 'District', 'Slide_Name', 'NH_SH_Location',
             'Latitude', 'Longitude', 'Material_Involved', 'Movement_Type', 'History']

data_rows = []
with pdfplumber.open("landslide_report_Assam.pdf") as pdf:
    for page in pdf.pages:
        for t in page.extract_tables():
            for row in t:
                # skip header rows and the title row that appears on page 1
                if row[0] in (None, 'Sl.No.') or row[0] is None:
                    continue
                if len(row) == len(col_names):
                    data_rows.append(row)

df = pd.DataFrame(data_rows, columns=col_names)
print("Total rows extracted:", len(df))

# Filter to Dima Hasao — the PDF uses a few different spellings for this district
# (Dima Hasao, Dima Hasao (N.C. Hills), Dima Hasao (NC Hills)), so match loosely
mask = df['District'].str.strip().str.lower().str.contains('dima hasao|nc hills|n.c. hills', na=False, regex=True)
dima_hasao_df = df[mask].copy()
print("Dima Hasao rows:", len(dima_hasao_df))

# Clean coordinates — some entries have blanks, stray characters, or comments in the fields
dima_hasao_df['Latitude'] = pd.to_numeric(dima_hasao_df['Latitude'], errors='coerce')
dima_hasao_df['Longitude'] = pd.to_numeric(dima_hasao_df['Longitude'], errors='coerce')
dima_hasao_df = dima_hasao_df.dropna(subset=['Latitude', 'Longitude'])
print("Usable rows after cleaning coordinates:", len(dima_hasao_df))

# Save the raw filtered table too, useful to check by eye
dima_hasao_df.to_csv("dima_hasao_landslides_raw.csv", index=False)

# Convert to a GeoDataFrame of points
import geopandas as gpd
from shapely.geometry import Point

geometry = [Point(xy) for xy in zip(dima_hasao_df['Longitude'], dima_hasao_df['Latitude'])]
landslides = gpd.GeoDataFrame(dima_hasao_df, geometry=geometry, crs="EPSG:4326")
landslides = landslides.to_crs(dst_crs)  # dst_crs = EPSG:32646, set back in Cell 4
landslides["label"] = 1

print("Final landslide point count for Dima Hasao:", len(landslides))

In [ ]:
# Cache the full extracted table so re-running never re-parses the 904-page PDF again
df.to_csv("gsi_full_inventory_cache.csv", index=False)

In [ ]:
!pip install pdfplumber -q

import pdfplumber

with pdfplumber.open("landslide_report_Assam.pdf") as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if tables:
            print(f"\n--- Page {i+1}: found {len(tables)} table(s) ---")
            for t in tables:
                for row in t[:3]:   # just first 3 rows as a preview
                    print(row)

In [ ]:
import osmnx as ox

roads_graph = ox.graph_from_polygon(boundary.geometry.iloc[0], network_type="drive")
roads_gdf = ox.graph_to_gdfs(roads_graph, nodes=False)
roads_gdf.to_file("roads.geojson", driver="GeoJSON")

print("Roads extracted:", len(roads_gdf), "segments")


In [ ]:
import numpy as np
from shapely.geometry import Point
import geopandas as gpd

boundary_utm = boundary.to_crs(dst_crs)
minx, miny, maxx, maxy = boundary_utm.total_bounds
n = len(landslides)

random_points = []
while len(random_points) < n:
    p = Point(np.random.uniform(minx, maxx), np.random.uniform(miny, maxy))
    if boundary_utm.contains(p).any():
        random_points.append(p)

negatives = gpd.GeoDataFrame(geometry=random_points, crs=dst_crs)
negatives["label"] = 0

print("Negative points generated:", len(negatives))

In [ ]:
import pandas as pd

points = gpd.GeoDataFrame(
    pd.concat([landslides[["geometry", "label"]], negatives[["geometry", "label"]]], ignore_index=True),
    crs=dst_crs
)

print("Total training points:", len(points))
print("Label counts:", points["label"].value_counts().to_dict())


In [ ]:
from rasterstats import point_query

points["elevation"] = point_query(points, "dem_reprojected.tif")
points["slope"] = point_query(points, "slope.tif")
points["aspect"] = point_query(points, "aspect.tif")

print(points[["elevation", "slope", "aspect"]].describe())

In [ ]:
roads = gpd.read_file("roads.geojson").to_crs(dst_crs)
points = gpd.sjoin_nearest(points, roads, distance_col="dist_to_road")

print(points["dist_to_road"].describe())


In [ ]:
import xarray as xr

ds = xr.open_dataset("RF25_ind2024_rfp25.nc")
print(ds)


In [ ]:
RAIN_VAR = "RAINFALL"
LAT_NAME = "LATITUDE"
LON_NAME = "LONGITUDE"

time_dim = "TIME" if "TIME" in ds.dims else ("time" if "time" in ds.dims else None)
rain_mean = ds[RAIN_VAR].mean(dim=time_dim) if time_dim else ds[RAIN_VAR]

points_latlon = points.to_crs("EPSG:4326")

rainfall_values = []
for geom in points_latlon.geometry:
    val = rain_mean.sel({LAT_NAME: geom.y, LON_NAME: geom.x}, method="nearest").values.item()
    rainfall_values.append(val)

points["rainfall"] = rainfall_values

print(points["rainfall"].describe())

In [ ]:
points.drop(columns=["geometry", "index_right"], errors="ignore").to_csv("landslide_points.csv", index=False)

print("Saved landslide_points.csv with", len(points), "rows")
print(points.head())